CAPA SILVER

En esta fase se cargan los datos que se han generado en bronze.

Se marcan las variables que nos quedamos, se descartan filas (si es necesario), se marcaran las banderas de calidad (en ningún momento se eliminan resgistros), añadir nombres de las zonas.

In [ ]:
!pip install -q pyspark
import os, re, glob
from functools import reduce
import pandas as pd
from pyspark.sql import SparkSession, functions as F
spark = (SparkSession.builder
         .master("local[*]")
         .appName("bronze")
         .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
         .getOrCreate())

In [ ]:
# Conectar a la carpeta del drive
from google.colab import drive

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/nyc-taxi"

Mounted at /content/drive


In [ ]:
# cargar datos de capa bronze
silver = spark.read.parquet(f"{BASE}/bronze")
silver.printSchema()

root
 |-- vendorid: integer (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- ratecodeid: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: integer (nullable = true)
 |-- dolocationid: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingested_at: timestamp (nullable = tru

In [ ]:
# cargar datos para la unión de los nombres de las zonas
zonas = spark.read.csv(f"{BASE}/taxi_zone_lookup.csv", header=True, inferSchema=True)
zonas.printSchema()
zonas.show(5)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [ ]:
# Umbrales para banderas de calidad

max_distance = 180
max_passengers = 7
MAX_MINUTES = 300

In [ ]:
# Columnas que se van a utilizar

# Listado de columnas que no se van a utilizar: vendedorid, ratecodeid, extra, mta_tax,improvement_surcharge, congestion_surcharge, cbd_congestion_fee, airport_fee, ehail_fee
quitar = ["vendorid", "ratecodeid", "extra", "mta_tax", "improvement_surcharge", "congestion_surcharge", "cbd_congestion_fee", "airport_fee", "ehail_fee"]
#mirar que todas existan
print(set(quitar) - set(silver.columns))
print(silver.columns)

silver = silver.drop(*quitar)
silver.printSchema()


set()
['vendorid', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'ratecodeid', 'store_and_fwd_flag', 'pulocationid', 'dolocationid', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee', 'source_file', 'ingested_at', 'ehail_fee', 'trip_type', 'taxi_type', 'year_month']
root
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- pulocationid: integer (nullable = true)
 |-- dolocationid: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- source_f

In [ ]:
# Mirar duplicados

variables = ["taxi_type", "pickup_datetime", "dropoff_datetime",
          "pulocationid", "dolocationid", "total_amount"]

n = silver.count()
n_unicos = silver.dropDuplicates(variables).count()
print(n, n_unicos, n - n_unicos)


3523552 3523551 1


In [ ]:
# mirar observaciones fuera del umbral del mes.
(silver.groupBy("taxi_type", "year_month")
       .agg(F.min("pickup_datetime").alias("primer_inicio"),
            F.max("pickup_datetime").alias("ultimo_inicio"),
            F.min("dropoff_datetime").alias("primer_fin"),
            F.max("dropoff_datetime").alias("ultimo_fin"))
       .show(truncate=False))


+---------+----------+-------------------+-------------------+-------------------+-------------------+
|taxi_type|year_month|primer_inicio      |ultimo_inicio      |primer_fin         |ultimo_fin         |
+---------+----------+-------------------+-------------------+-------------------+-------------------+
|yellow   |2025-01   |2024-12-31 20:47:55|2025-02-01 00:00:44|2024-12-18 07:52:40|2025-02-01 23:44:11|
|grenn    |2025-01   |2024-12-25 23:13:15|2025-02-05 18:46:24|2024-12-25 23:13:17|2025-02-05 19:11:47|
+---------+----------+-------------------+-------------------+-------------------+-------------------+



In [ ]:
# para ver cuantos casos son
(silver.withColumn("mes_recogida", F.date_format("pickup_datetime", "yyyy-MM"))
       .filter(F.col("mes_recogida") != F.col("year_month"))
       .groupBy("taxi_type", "year_month", "mes_recogida").count()
       .orderBy("taxi_type", "mes_recogida")
       .show())

+---------+----------+------------+-----+
|taxi_type|year_month|mes_recogida|count|
+---------+----------+------------+-----+
|    grenn|   2025-01|     2024-12|    6|
|    grenn|   2025-01|     2025-02|   37|
|   yellow|   2025-01|     2024-12|   21|
|   yellow|   2025-01|     2025-02|    1|
+---------+----------+------------+-----+



Ha observaciones con fecha muy lejana al mes que debería de ser, por ello se decide eliminar según un criterio.

*Criterio: si hay una observación que empezó el viaje en el mes anterior pero lo acaba en el qu etoca, todo ok. Si lo empezó en el mes que toca y acaba en el mes que toca todo OK. Si empieza en el mes que toca y acaba en el siguiente todo Ok. si no cumple esto no se tiene en cuenta y se elimina.*

In [ ]:
# mirar cuantos se eliminan según el criterio
mes_fichero = F.to_date(F.concat(F.col("year_month"), F.lit("-01")))
mes_ini = F.trunc("pickup_datetime", "month")
mes_fin = F.trunc("dropoff_datetime", "month")
criterio = (
    ((mes_ini == mes_fichero) & (mes_fin == mes_fichero)) |
    ((mes_ini == F.add_months(mes_fichero, -1)) & (mes_fin == mes_fichero)) |
    ((mes_ini == mes_fichero) & (mes_fin == F.add_months(mes_fichero, 1)))
)
n_antes = silver.count()

(silver.withColumn("cumple", criterio)
       .groupBy("taxi_type", "cumple").count()
       .show())

silver = silver.filter(criterio)
print(n_antes, silver.count(), n_antes - silver.count())


+---------+------+-------+
|taxi_type|cumple|  count|
+---------+------+-------+
|   yellow|  true|3475211|
|   yellow| false|     15|
|    grenn| false|     43|
|    grenn|  true|  48283|
+---------+------+-------+

3523552 3523494 58


fuera de mes: 15 en amarillos y 43 en verdes.

In [ ]:
# Añadir columna de tiempo del trayecto (en minutos)
silver = silver.withColumn(
    "trip_minutes",
    F.round((F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60,2)
)

#mirar que cuadre
silver.select("pickup_datetime", "dropoff_datetime", "trip_minutes").show(5, truncate=False)

+-------------------+-------------------+------------+
|pickup_datetime    |dropoff_datetime   |trip_minutes|
+-------------------+-------------------+------------+
|2025-01-01 00:18:38|2025-01-01 00:26:59|8.35        |
|2025-01-23 17:13:47|2025-01-23 17:20:30|6.72        |
|2025-01-01 00:32:40|2025-01-01 00:35:13|2.55        |
|2025-01-23 17:24:39|2025-01-23 17:43:19|18.67       |
|2025-01-01 00:44:04|2025-01-01 00:46:01|1.95        |
+-------------------+-------------------+------------+
only showing top 5 rows


In [ ]:
# Saber cuantos viajes hay de minutos menos de 0 y más de 300 minutos.
(silver.groupBy("taxi_type")
       .agg(F.min("trip_minutes").alias("minimo"),
            F.expr("percentile_approx(trip_minutes, 0.5)").alias("mediana"),
            F.expr("percentile_approx(trip_minutes, 0.99)").alias("p99"),
            F.max("trip_minutes").alias("maximo"),
            F.count(F.when(F.col("trip_minutes") <= 0, 1)).alias("cero_o_negativa"),
            F.count(F.when(F.col("trip_minutes") > 300, 1)).alias("mas_de_5h"))
       .show())

+---------+------+-------+-----+-------+---------------+---------+
|taxi_type|minimo|mediana|  p99| maximo|cero_o_negativa|mas_de_5h|
+---------+------+-------+-----+-------+---------------+---------+
|   yellow|-56.03|   11.7|58.97|5626.32|           2050|     1233|
|    grenn|   0.0|  11.25| 61.9|1439.65|             48|      235|
+---------+------+-------+-----+-------+---------------+---------+



Limitar el tiempo de los viajes y solo aceptar aquellos con más de 0 y menos de 300 min.

Banderas de calidad para la capa oro

In [ ]:
# Viajes con "total_amount < 0". Son reembolsos, anulaciones o ajustes, no viajes reales. Si se contaran, duplicarían viajes y restarían ingresos (153 en verdes, 63.037 en amarillos)
silver = silver.withColumn("is_adjustment", F.col("total_amount") < 0)

# Se excluye de demanda e ingresos

In [ ]:
# Tarifa negativa pero total postitivo. El desglose de la tarifa es incoherente con el total (81.402 amarillos, casi todos del tipo de pago 0; 0 en verdes)
silver = silver.withColumn("fare_unreliable", F.col("fare_amount") < 0)

# en estos casos no se usa fare_amount sino total_amount

In [ ]:
# Pasajeros nulos, 0 o 7 o más. Al ser un dato que s eintroduce manual se descartan esos casos para los calculos con pasajeros.
silver = silver.withColumn(
    "passenger_unknown",
    F.col("passenger_count").isNull()
    | (F.col("passenger_count") == 0)
    | (F.col("passenger_count") > max_passengers)
)

# Esos casos no se utilizaran para el análisis de pasajeros

In [ ]:
# Tipo de pago = 0, es un segmento distinto (FLEX FARE) que distorsiona medias.
silver = silver.withColumn("is_flex_fare", F.col("payment_type") == 0)

# Se muestra aparte en algunas preguntas de negocio y se excluyen de propinas

In [ ]:
# Limitar duración del trayecto

silver = silver.withColumn(
    "duration_valid",
    (F.col("trip_minutes") > 0) & (F.col("trip_minutes") < MAX_MINUTES)
)

In [ ]:
# Limitar distancia recorrida
silver = silver.withColumn(
    "distance_valid",
    (F.col("trip_distance") > 0) & (F.col("trip_distance") < max_distance)
)

Unir los nombres de las zonas de origen y destino de los trayectos

In [ ]:
#renombrar las columnas en dos ficheros distintos, para tener distintos nombres en origen y destino.
zonas_pu = zonas.select(
    F.col("locationid").alias("pulocationid"),
    F.col("borough").alias("pickup_borough"),
    F.col("zone").alias("pickup_zone"),
)
zonas_do = zonas.select(
    F.col("locationid").alias("dolocationid"),
    F.col("borough").alias("dropoff_borough"),
    F.col("zone").alias("dropoff_zone"),
)

In [ ]:
# union de datos
n_antes = silver.count()

silver = (silver
          .join(F.broadcast(zonas_pu), on="pulocationid", how="left")
          .join(F.broadcast(zonas_do), on="dolocationid", how="left"))

print(n_antes, silver.count())

3523494 3523494


In [ ]:
# Zonas aeropuerto
aeropuerto_zonas = [1, 132, 138]   # EWR, JFK, LGA

silver = silver.withColumn(
    "is_airport",
    F.col("pulocationid").isin(aeropuerto_zonas) | F.col("dolocationid").isin(aeropuerto_zonas)
)

In [ ]:
# Comporbación todo ok en los joins
silver.filter(F.col("pickup_zone").isNull() | F.col("dropoff_zone").isNull()).count()

0

In [ ]:
# Comprobación aeropuerto
silver.groupBy("taxi_type", "is_airport").count().show()

+---------+----------+-------+
|taxi_type|is_airport|  count|
+---------+----------+-------+
|   yellow|      true| 293405|
|   yellow|     false|3181806|
|    grenn|     false|  46850|
|    grenn|      true|   1433|
+---------+----------+-------+



Guardar en silver

In [ ]:
(silver.write
       .mode("overwrite")
       .partitionBy("taxi_type", "year_month")
       .parquet(f"{BASE}/silver"))

Tiempo de ejecución.

Cargar datos de bronze --> 7s.

Mirar duplicados --> 27s.

Mirar fechas --> 3s.

Mirar que fechas no tener según el criterio --> 12s.

Creación de banderas de calidad --> 0s.

Joins --> 10s.

Guardar datos capa silver --> 43s.